# Notebook 01 — LLM as a judge (and when to trust it)

**Workshop:** Agentic AI for Scientists — Week 5 (Evaluation & Benchmarking)
**Block:** model-graded evaluation (20 min)
**Goal:** Many qualities you care about — *faithfulness, helpfulness, clinical safety, tone* — have no clean F1. The modern answer is to use a strong LLM as a **judge**. We build a pointwise judge and a pairwise judge, then spend real time on the part most tutorials skip: **how judges are biased and how to defend against it.**

**Two judge designs:**
- **Pointwise** — score one answer on a rubric (1–5). Simple, but scores drift between runs.
- **Pairwise** — "is A better than B?" More reliable (relative is easier than absolute), and it's how leaderboards like Chatbot Arena actually work.

**Why care about bias:** an LLM judge has known failure modes — **position bias** (favors the first option), **verbosity bias** (longer = better), **self-preference** (favors its own family's outputs). If you don't measure these, your eval is quietly wrong.

## 0. Setup

In [ ]:
%pip install -q "langchain-google-genai>=2.0" pydantic

In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab : add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local : put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except Exception:
        pass
if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("Gemini API key loaded.")

---
## 1. A pointwise judge with a rubric

The rubric is the whole game. A vague "rate 1–5" gives noisy garbage; an explicit rubric with anchored levels gives consistency. We bind a Pydantic schema so the judge returns a score **and its reasoning** (always read the reasoning — a right score for a wrong reason is a red flag).

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

class Verdict(BaseModel):
    reasoning: str = Field(description="1-2 sentences justifying the score, citing the rubric.")
    score: int = Field(description="Integer 1-5 per the rubric.", ge=1, le=5)

RUBRIC = """Score a clinical answer 1-5 for FAITHFULNESS + HELPFULNESS:
5 = clinically accurate, directly answers, no invented facts.
4 = accurate and helpful, minor omission.
3 = mostly correct but vague or missing a key point.
2 = partially incorrect or unhelpful.
1 = wrong, unsafe, or fabricated (hallucinated drug/diagnosis)."""

pointwise = judge_llm.with_structured_output(Verdict)

def judge_pointwise(question, answer):
    prompt = f"{RUBRIC}\n\nQUESTION:\n{question}\n\nANSWER:\n{answer}\n\nScore it."
    return pointwise.invoke(prompt)

v = judge_pointwise(
    "What are the symptoms of Hashimoto's disease?",
    '{"disease": "Hashimoto\'s disease", "symptoms": ["fatigue", "weight gain", "cold intolerance"], "treatment": ["levothyroxine"]}')
print(f"score={v.score}  reason={v.reasoning}")

In [ ]:
# Contrast: judge a deliberately bad (hallucinated) answer — a good judge should punish it.
bad = '{"disease": "Hashimoto\'s disease", "symptoms": ["fever", "rash"], "treatment": ["amoxicillin", "chemotherapy"]}'
vb = judge_pointwise("What are the symptoms of Hashimoto's disease?", bad)
print(f"score={vb.score}  reason={vb.reasoning}")
print("\nA judge that scores the hallucinated antibiotic+chemo answer >=3 is not safe to use as-is.")

---
## 2. A pairwise judge — and measuring position bias

Pairwise is more reliable, but it's where **position bias** bites: judges over-pick whichever answer comes first. The defense is simple and non-negotiable — **run both orders and average**. If A beats B in one order and B beats A in the other, the judge is guessing.

In [ ]:
class Pair(BaseModel):
    reasoning: str = Field(description="Why the winner is better.")
    winner: str = Field(description="'A' or 'B'.")

pairwise = judge_llm.with_structured_output(Pair)

def judge_pairwise(question, ans_a, ans_b):
    prompt = (f"Which answer is better for the question? Reply with the winner.\n\n"
              f"QUESTION:\n{question}\n\nANSWER A:\n{ans_a}\n\nANSWER B:\n{ans_b}")
    return pairwise.invoke(prompt).winner

def judge_pairwise_debiased(question, ans_a, ans_b):
    """Run both orders. Returns 'A', 'B', or 'TIE' (disagreement = no reliable winner)."""
    w1 = judge_pairwise(question, ans_a, ans_b)            # A first
    w2 = judge_pairwise(question, ans_b, ans_a)            # B first (labels swap)
    w2 = "A" if w2 == "B" else "B"                          # un-swap back to A/B identity
    return w1 if w1 == w2 else "TIE"

q = "What are the symptoms of Hashimoto's disease?"
good = '{"disease":"Hashimoto disease","symptoms":["fatigue","weight gain","cold intolerance"],"treatment":["levothyroxine"]}'
weak = '{"disease":"thyroid problem","symptoms":["tired"],"treatment":[]}'
print("naive (A=good, B=weak):", judge_pairwise(q, good, weak))
print("naive (A=weak, B=good):", judge_pairwise(q, weak, good))
print("debiased verdict      :", judge_pairwise_debiased(q, good, weak))

---
## 3. Calibrate the judge against humans

**An unvalidated judge is just a vibe.** Before trusting a judge at scale, check its agreement with a few human labels. Even 10–20 hand-scored examples tell you whether the judge tracks your judgment. Here we show the pattern with a tiny labeled set; in practice you'd score ~50 yourself.

In [ ]:
# A handful of (answer, human_score) pairs you labeled. Does the judge agree?
HUMAN_LABELED = [
    ("Hashimoto's: fatigue, weight gain, cold intolerance; treat with levothyroxine.", 5),
    ("It's a thyroid thing, you feel tired.", 2),
    ("Hashimoto's disease is treated with antibiotics and chemotherapy.", 1),
    ("Symptoms include fatigue and weight gain. See a doctor for treatment.", 3),
]
q = "What are the symptoms and treatment of Hashimoto's disease?"
agree, errs = 0, []
for ans, human in HUMAN_LABELED:
    j = judge_pointwise(q, ans).score
    errs.append(abs(j - human))
    agree += (abs(j - human) <= 1)   # within 1 point = agreement
print(f"within-1-point agreement: {agree}/{len(HUMAN_LABELED)}")
print(f"mean absolute error      : {sum(errs)/len(errs):.2f}")
print("\nIf MAE is ~>1.5, fix the rubric (add anchors/examples) before scaling this judge up.")

## What you learned

A judge is a measuring instrument, and instruments need calibration. You built pointwise + pairwise judges, neutralized position bias by averaging orders, and checked agreement against human labels.

**Rules of thumb to keep:** prefer pairwise for reliability; always swap orders; always read the reasoning; validate against humans before trusting at scale; never let a model judge be the *only* gate on something safety-critical.

**Next — `02_benchmarks.ipynb`:** step up from your own data to **published benchmarks** — run a real medical QA benchmark, and map the agent-benchmark landscape (SWE-bench, GAIA, AgentBench).